# 05 — Training Data Preparation

Assemble the final training dataset from:
1. Real-world pairs: WDC JSON-LD + CC HTML + Playwright screenshots
2. Synthetic pairs: Claude-generated JSON-LD (from notebook 06) for pages without schema

Applies quality filtering, deduplication, and type balancing.
Splits into train/eval sets.

**Target**: 10,000-50,000 high-quality (screenshot + HTML → JSON-LD) pairs

**Outputs**:
- `data/processed/dataset.jsonl` — full assembled dataset
- `data/processed/train.jsonl` — 90% split
- `data/processed/eval.jsonl` — 10% split

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import Counter

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

from src.training_data import format_training_example, assemble_dataset, split_dataset, dataset_stats
from src.schema_validator import validate_jsonld
print('Imports OK')

## Load All Source Data

In [ ]:
# Load WARC manifest (html + wdc_jsonld + screenshot_path)
manifest = []
manifest_path = Path('../data/raw/warc_manifest.jsonl')
if manifest_path.exists():
    with open(manifest_path) as f:
        for line in f:
            manifest.append(json.loads(line))
    print(f'Manifest entries: {len(manifest)}')
else:
    print('No manifest found — run notebooks 02-04 first')

# Load synthetic data (from notebook 06, if run)
synthetic_records = []
synth_dir = Path('../data/processed/synthetic')
if synth_dir.exists():
    for f in synth_dir.glob('*.json'):
        with open(f) as fh:
            rec = json.load(fh)
        if rec.get('valid') and rec.get('generated_schema'):
            synthetic_records.append(rec)
    print(f'Synthetic records: {len(synthetic_records)}')
else:
    print('No synthetic data yet (run notebook 06 to generate)')

## Quality Filtering Pipeline

In [ ]:
MIN_QUALITY = 0.3
MIN_PROPERTY_COUNT = 5

valid_examples = []
stats = Counter()

for entry in manifest:
    # Load HTML
    html_path = entry.get('html_path')
    if not html_path or not Path(html_path).exists():
        stats['no_html'] += 1
        continue
    
    with open(html_path, 'r', errors='replace') as f:
        html = f.read()
    
    if len(html) < 200:
        stats['html_too_short'] += 1
        continue
    
    # Validate JSON-LD
    wdc_jsonld = entry.get('wdc_jsonld')
    if not wdc_jsonld:
        stats['no_jsonld'] += 1
        continue
    
    jsonld_str = json.dumps(wdc_jsonld, ensure_ascii=False)
    validation = validate_jsonld(jsonld_str)
    
    if not validation['valid']:
        stats['invalid_jsonld'] += 1
        continue
    
    if validation['quality_score'] < MIN_QUALITY:
        stats['low_quality'] += 1
        continue
    
    stats['accepted'] += 1
    valid_examples.append({
        'html': html,
        'jsonld': wdc_jsonld,
        'screenshot_path': entry.get('screenshot_path'),
        'source': 'wdc',
        'schema_types': validation['schema_types'],
        'domain': entry.get('url', '').split('/')[2] if entry.get('url') else '',
        'quality_score': validation['quality_score'],
    })

print('Filtering stats:')
for k, v in sorted(stats.items()):
    print(f'  {k}: {v}')

In [ ]:
# Add synthetic examples
for rec in synthetic_records:
    valid_examples.append({
        'html': rec.get('html', ''),
        'jsonld': rec.get('generated_schema'),
        'screenshot_path': rec.get('screenshot_path'),
        'source': 'synthetic',
        'schema_types': rec.get('schema_types', []),
        'domain': rec.get('source_url', '').split('/')[2] if rec.get('source_url') else '',
        'quality_score': rec.get('quality_score', 0.0),
    })

print(f'Total examples before deduplication: {len(valid_examples)}')
print(f'  WDC: {sum(1 for e in valid_examples if e["source"] == "wdc")}')
print(f'  Synthetic: {sum(1 for e in valid_examples if e["source"] == "synthetic")}')

## Deduplication and Type Balancing

In [ ]:
import random

# Deduplicate by domain
seen_domains = set()
deduped = []
for ex in valid_examples:
    domain = ex['domain']
    if domain not in seen_domains:
        seen_domains.add(domain)
        deduped.append(ex)

print(f'After domain deduplication: {len(deduped)} examples')

# Check type distribution
type_counts = Counter()
for ex in deduped:
    for t in ex.get('schema_types', ['Unknown']):
        type_counts[t] += 1

print('\nSchema type distribution:')
for t, count in type_counts.most_common(15):
    print(f'  {t}: {count}')

In [ ]:
# Balance: cap overrepresented types at 5000 examples each
MAX_PER_TYPE = 5_000

type_seen = Counter()
balanced = []

random.shuffle(deduped)
for ex in deduped:
    primary_type = ex.get('schema_types', ['Unknown'])[0] if ex.get('schema_types') else 'Unknown'
    if type_seen[primary_type] < MAX_PER_TYPE:
        type_seen[primary_type] += 1
        balanced.append(ex)

print(f'After balancing: {len(balanced)} examples')

## Assemble and Save Dataset

In [ ]:
import uuid

dataset_path = PROCESSED_DIR / 'dataset.jsonl'

with open(dataset_path, 'w') as f:
    for i, ex in enumerate(balanced):
        example = format_training_example(
            html=ex['html'],
            jsonld=ex['jsonld'],
            screenshot_path=ex.get('screenshot_path'),
            example_id=f"train_{i:06d}",
            source=ex['source'],
            schema_types=ex.get('schema_types'),
            domain=ex.get('domain'),
            quality_score=ex.get('quality_score'),
        )
        f.write(json.dumps(example, ensure_ascii=False) + '\n')

print(f'Saved {len(balanced)} examples to {dataset_path}')

In [ ]:
# Train/eval split
train_count, eval_count = split_dataset(
    input_path=str(dataset_path),
    train_path=str(PROCESSED_DIR / 'train.jsonl'),
    eval_path=str(PROCESSED_DIR / 'eval.jsonl'),
    train_ratio=0.9,
)
print(f'Split: {train_count} train, {eval_count} eval')

In [ ]:
# Final stats
stats = dataset_stats(str(dataset_path))
print('\nDataset statistics:')
for k, v in stats.items():
    if k != 'schema_type_distribution':
        print(f'  {k}: {v}')
print('\nType distribution:')
for t, count in list(stats['schema_type_distribution'].items())[:10]:
    print(f'  {t}: {count}')

print(f'\nReady for fine-tuning!')
print('Next step: 06_synthetic_data_gen.ipynb (to add more examples) or 07_fine_tuning.ipynb')